#How This Script Works
* Function 1 (identify_main_intent):
Uses regex searches on the SERP features text to assign one or more intent categories (e.g., Informational, Transactional).

* Function 2 (identify_intent_and_platforms):
Leverages the main intent function and further inspects the SERP features to determine the desired content platform (e.g., Video, Article/Blog).

* Function 3 (classify_domain):
Classifies the primary domain for each keyword by checking for known domain indicators (e.g., Wikipedia, Amazon, G2).

##Main Script:
After the user uploads a CSV file, the script reads it into a DataFrame and applies each function to create new columns with the classifications.

**You can further customize the regex rules and the conditions within each function to better suit your SERP analysis and rule-based classification needs.**

In [ ]:
import pandas as pd
import re
import io
from google.colab import files

#Search Intent Rule-based classification from SERP features Function - How it Works
This notebook performs a rule-based classification of search intent from SERP features. Here’s what it does step-by-step:

1. File Upload:
You’ll be prompted to upload a CSV file that contains two columns:

* keyword: The search query.
* SERP Features: A text field listing various SERP features (e.g., "answer box", "carousel", "local pack", etc.) that appeared in the search results.
2. Mapping SERP Features to Intent Categories:
We use the function identify_main_intent_serp to map known SERP features to intent categories. For example:

* AI Overview and Answer Box → Informational
*Bottom Ads and Paid Results → Transactional
*GMB, Local Pack, Sitelinks → Navigational
*Carousel and Carousel results (or similar cues) → Commercial Investigation
If multiple features are present in the SERP features text, the function concatenates all the corresponding intent categories into one string.
3. Calculating Percentage Breakdown:
In addition to the combined intent label, we calculate the percentage contribution of each intent category (Informational, Navigational, Transactional, Commercial Investigation) for each keyword. The function calculate_intent_percentages:

* Searches the SERP features text for occurrences of each feature from a predefined mapping.
* Counts the total occurrences of all features.
* Calculates what percentage of those occurrences corresponds to each intent category.
* Adds four new columns to the DataFrame, one for each intent category, showing these percentages.
Saving and Downloading the Results:
* After processing, the notebook saves the updated DataFrame (which now includes the concatenated intent string and the four percentage columns) to a new CSV file. You’ll then be prompted to download this file for further analysis.

By following these steps, the notebook gives you both a combined intent classification per keyword (from the SERP features) and a detailed breakdown of the intent distribution across the four categories. This enables deeper insights into the search intent behind each query based on the features observed in the SERPs.

In [ ]:
# --- Code to upload file, process it, and output the results ---

from google.colab import files
import pandas as pd
import io

# Upload CSV file with columns "keyword" and "SERP features"
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Apply the provided function to map search intent categories (concatenated) into a new column "Search Intent"
df["Search Intent"] = df["SERP Features"].apply(identify_main_intent_serp)

# Define a function to calculate percentages for each intent category
def calculate_intent_percentages(serp_features):
    mapping = {
        "ai overview": "Informational",
        "answer box": "Informational",
        "app": "Transactional",
        "bottom ads": "Transactional",
        "carousel": "Commercial Investigation",
        "carousel results": "Commercial Investigation",
        "commercial units": "Transactional",
        "copyright infringement": "Unclassified",
        "currency box": "Informational",
        "events": "Informational",
        "explore brands": "Navigational",
        "faq": "Informational",
        "featured snippet": "Informational",
        "featured snippets": "Informational",
        "find results on": "Informational",
        "flights": "Transactional",
        "gmb": "Navigational",
        "google flights": "Transactional",
        "google hotels": "Transactional",
        "google posts": "Informational",
        "google reviews": "Commercial Investigation",
        "hotel pack": "Transactional",
        "image": "Informational",
        "images": "Informational",
        "jobs": "Transactional",
        "knowledge card": "Informational",
        "knowledge graph": "Informational",
        "local pack": "Navigational",
        "local services": "Transactional",
        "map": "Navigational",
        "math solver": "Informational",
        "mention carousel": "Informational",
        "multi carousel": "Commercial Investigation",
        "organic results": "Informational",
        "paid results": "Transactional",
        "people also ask": "Informational",
        "people also search for": "Informational",
        "podcasts": "Informational",
        "popular products": "Transactional",
        "positions.title.datasets": "Informational",
        "positions.title.discussions_and_forums": "Informational",
        "product consideration": "Commercial Investigation",
        "questions & answers": "Informational",
        "recipes": "Informational",
        "refine products": "Transactional",
        "related searches": "Informational",
        "reviews": "Commercial Investigation",
        "scholarly articles": "Informational",
        "see results about": "Informational",
        "shopping": "Transactional",
        "shopping results": "Transactional",
        "sitelinks": "Navigational",
        "sites": "Navigational",
        "sports": "Informational",
        "stocks box": "Informational",
        "things to do": "Informational",
        "top ads": "Transactional",
        "top sights": "Navigational",
        "top stories": "Informational",
        "twitter": "Informational",
        "video": "Informational",
        "videos block": "Informational",
        "visual stories": "Informational",
        "x (ex-twitter)": "Informational"
    }
    if not isinstance(serp_features, str):
        return {"Informational": 0, "Navigational": 0, "Transactional": 0, "Commercial Investigation": 0}
    text = serp_features.lower()
    counts = {"Informational": 0, "Navigational": 0, "Transactional": 0, "Commercial Investigation": 0}
    total = 0
    for feature, intent in mapping.items():
        cnt = text.count(feature)
        if cnt > 0:
            if intent in counts:
                counts[intent] += cnt
            total += cnt
    if total > 0:
        percentages = {key: (counts[key] / total) * 100 for key in counts}
    else:
        percentages = {key: 0 for key in counts}
    return percentages

# Calculate percentages for each row and add four new columns
intent_categories = ["Informational", "Navigational", "Transactional", "Commercial Investigation"]
percents_series = df["SERP Features"].apply(calculate_intent_percentages)
for cat in intent_categories:
    df[f"{cat} (%)"] = percents_series.apply(lambda d: d[cat])

# Save the updated DataFrame to a CSV file and download it
output_filename = "classified_SERP_intents.csv"
df.to_csv(output_filename, index=False)
files.download(output_filename)
df.head(10)


# Function 2: Map ranked websites on key positions to holistic intent categories - How it works


## (1) classify top website for main intent categories

* SITE_TYPE_RULES:

A list of tuples.
Each tuple has a list of keywords and a corresponding pair: (site_type, intent).
For example, if a domain contains any of ["wiki", "wikimedia", "wiktionary"], it is classified as ("Wiki", "Informational").
Order matters—the script uses the first matching rule and then stops.
classify_site_and_intent(domain):

* Iterates through SITE_TYPE_RULES in order.
Performs a substring check on the domain name.
Returns the first matching (Site Type, Holistic Intent Category).
Falls back to ("General", "Informational") if no rules match.
File Upload and Processing in Colab:

Uses files.upload() to let you select your CSV from your local system.
Reads the CSV into a Pandas DataFrame and checks for a Domain column.
Iterates over each domain, classifies it, and adds two new columns: Site Type and Holistic Intent Category.
* Resulting CSV:

Saved as classified_domains.csv.
Automatically downloaded via files.download().
You can refine or reorder the SITE_TYPE_RULES list as you learn more about each domain or wish to tailor the site types and intent categories. This approach provides a straightforward rule-based classification that you can easily expand.

## (2) classify domains from SERP analysis into holistic search intent categories they serve

* File Upload:
The script prompts you to upload both the classified_domains.csv (with columns such as "Domain" and "Holistic Intent Category") and the SERP results file.

* Domain Extraction:
The extract_domain_from_url function uses a regex to pull the domain from each URL.

* Lookup Dictionary:
The script iterates through df_domains and creates a dictionary (domain_map) where each key is the lowercased domain and its value is the corresponding "Holistic Intent Category".

* Classification:
For each row in the SERP results DataFrame, it extracts the domain and looks up the holistic intent using the lookup dictionary. If no match is found, it returns "Unclassified".

* Output:
The updated DataFrame is saved as SERP_results_with_holistic_intent.csv and automatically downloaded.


In [ ]:
#before this ---- classify top 1000 domains on intent

import pandas as pd
import io
import re
from google.colab import files

# A list of (keywords, (site_type, intent)) tuples.
# Order matters: the first match wins. You can refine or expand this as needed.
SITE_TYPE_RULES = [
    (["wiki", "wikimedia", "wiktionary"], ("Wiki", "Informational")),
    (["udemy","coursera","khanacademy","edx","tutorialspoint","toppr","byjus","study.com","unacademy"], ("Courses/Learning", "Informational")),
    (["news","times","cnn","bbc","nytimes","hindustantimes","reuters","theguardian","washingtonpost","npr","cnbc","bloomberg","forbes","newsweek","abcnews","indiatimes"], ("News", "Informational")),
    (["reviews","trustpilot","sitejabber","resellerratings","g2","capterra","pitchbook"], ("Reviews", "Commercial")),
    (["techradar","pcmag","tomshardware","slashgear","engadget","gizmodo","theverge","wired","cnet","zdnet","techspot","techtarget"], ("Technology", "Informational")),
    (["shopping","shopee","mercadolibre","temu","etsy","ebay","flipkart","rakuten","walmart","target","bestbuy","sportsdirect","amazon","alibaba","aliexpress","shein","asos"], ("Shopping/E-commerce", "Transactional")),
    (["facebook","instagram","twitter","x.com","linkedin","tiktok","snapchat","pinterest","reddit","quora","tumblr","wechat","medium","stackexchange"], ("Social Media / Forum", "Commercial Investigation")),
    (["dictionary","linguee","wordreference","merriam-webster","collinsdictionary","cambridge","urbandictionary","lexico","yourdictionary","ldoceonline","bab.la","reverso"], ("Dictionary/Language", "Informational")),
    ([".gov",".edu",".ac.",".gouv"], ("Government/Education", "Informational")),
    (["health","webmd","mayoclinic","clevelandclinic","healthline","nih","cdc","hhs","nhs","fda","medicinenet","medicalnewstoday","verywellhealth","medlineplus"], ("Health", "Informational")),
    (["tripadvisor","expedia","booking","trivago","airbnb","hotels","lonelyplanet","trip.com","agoda","rome2rio","travelocity","fodor","kayak","makemytrip","orbitz"], ("Travel", "Commercial")),
    (["indeed","monster","ziprecruiter","glassdoor","linkedinjobs","naukri","job","simplyhired"], ("Jobs", "Commercial")),
    (["finance","moneycontrol","investing","bankrate","wsj","bloomberg","marketwatch","nasdaq","seekingalpha"], ("Finance", "Commercial Investigation")),
    (["imdb","rottentomatoes","screenrant","gamespot","ign","metacritic","letterboxd","fandom","spotify","soundcloud","deezer","iheartradio","disneyplus","netflix","hulu","primevideo","hbomax"], ("Entertainment", "Informational")),
    (["buzzfeed","popsugar","cosmopolitan","elle","vogue","harpersbazaar","menshealth","womenshealth","goodhousekeeping","marieclaire","instyle","glamour","bhg","nymag","vanityfair"], ("Lifestyle", "Informational")),
    (["allrecipes","food","recipes","yummly","epicurious","foodnetwork","tasteofhome","thekitchn","cookinglight","bonappetit","bbcgoodfood"], ("Food", "Informational")),
    (["espn","bleacherreport","skysports","sportsdirect","cbssports","si","goal","foxsports","nba","mlb","nhl","nfl","fifa"], ("Sports", "Informational")),
    (["mapquest","maps.google","waze","mapcarta","moovitapp","here.com","tomtom"], ("Maps/Navigation", "Navigational")),
    (["stanford","mit.","harvard","berkeley","oxford","cambridge","researchgate","jstor","sciencedirect","springer","tandfonline","bmj","nasa","edu","ac."], ("Academic/Research", "Informational")),
    (["github","gitlab","sourceforge"], ("Developer Platform", "Informational")),
]

def classify_site_and_intent(domain):
    """
    Returns (Site Type, Holistic Intent Category) based on the rules in SITE_TYPE_RULES.
    If no rule matches, defaults to ("General", "Informational").
    """
    if not isinstance(domain, str):
        return ("Unclassified", "Unclassified")

    d = domain.lower()

    # Check each rule in order
    for keywords, (site_type, intent) in SITE_TYPE_RULES:
        # If the domain or subdomain matches any keyword, return that classification
        # We do a simple substring check; adjust as needed for stricter matching.
        if any(k in d for k in keywords):
            return (site_type, intent)

    # Fallback
    return ("General", "Informational")

# Prompt the user to upload the CSV file
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Load the CSV into a DataFrame
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Make sure the CSV has a column named "Domain"
if "Domain" not in df.columns:
    raise ValueError("The CSV must contain a 'Domain' column.")

# Classify each domain
site_types = []
holistic_intents = []

for domain in df["Domain"]:
    site_type, intent = classify_site_and_intent(domain)
    site_types.append(site_type)
    holistic_intents.append(intent)

# Add new columns to the DataFrame
df["Site Type"] = site_types
df["Holistic Intent Category"] = holistic_intents

# Save and download the updated CSV
output_filename = "classified_domains.csv"
df.to_csv(output_filename, index=False)
files.download(output_filename)


In [ ]:
import pandas as pd
import io
import re
from google.colab import files

def extract_domain_from_url(url):
    """
    Extracts the domain from a given URL.
    For example:
      - https://www.youtube.com/watch?v=123 -> youtube.com
      - https://sub.domain.amazon.co.uk/gp -> amazon.co.uk
    """
    if not isinstance(url, str) or not url.strip():
        return ""
    match = re.search(r"https?://(?:www\.)?([^/]+)", url.lower())
    if match:
        return match.group(1)
    return ""

def classify_domain_from_lookup(domain, domain_map):
    """
    Looks up a domain in the domain_map dictionary to find its 'Holistic Intent Category'.
    Returns "Unclassified" if no match is found.
    """
    return domain_map.get(domain, "Unclassified")

# 1) Upload the two files in Colab:
print("Please upload 'classified_domains.csv' ...")
uploaded_domains = files.upload()
print("Now upload 'SAMPLE-serps-results-20241117-162617.csv' ...")
uploaded_serps = files.upload()

# 2) Read the files into DataFrames
domains_filename = list(uploaded_domains.keys())[0]
serps_filename = list(uploaded_serps.keys())[0]

df_domains = pd.read_csv(io.BytesIO(uploaded_domains[domains_filename]))
df_serps = pd.read_csv(io.BytesIO(uploaded_serps[serps_filename]))

# 3) Build a lookup dictionary using the correct column names:
#    Columns in classified_domains.csv: Rank, Domain, Keywords in SERPs, Estimated organic traffic, Site Type, Holistic Intent Category
domain_map = {}
for i, row in df_domains.iterrows():
    # Use the "Domain" as key and "Holistic Intent Category" as value
    domain_map[row["Domain"].lower()] = row["Holistic Intent Category"]

# 4) Extract the domain from each SERP result URL and look up the classification
df_serps["extracted_domain"] = df_serps["url"].apply(extract_domain_from_url)
df_serps["holistic search intent per domain"] = df_serps["extracted_domain"].apply(
    lambda d: classify_domain_from_lookup(d, domain_map)
)

# 5) Save the updated SERP results to a new CSV and prompt download
output_filename = "SERP_results_with_holistic_intent.csv"
df_serps.to_csv(output_filename, index=False)
files.download(output_filename)

print("Done! The updated CSV has been downloaded.")